# 14 - Encode external diagnoses, reports, and normalized concepts

BiomedCLIP supplies a frozen biomedical text representation. Two text views are written:

- `raw`: the complete machine diagnosis or CXR report.
- `concept`: a short assertion-aware concept list using `present:`, `absent:`, and `uncertain:` prefixes.

The default concept extractor is deterministic and auditable. To ablate LLM reasoning, provide `LLM_CONCEPTS_JSONL`, one JSON object per line with `sample_id` and a list of assertion-aware `concepts`. LLM outputs are treated as inferred semantic supervision, never as encounter ground truth.

Text embeddings are written incrementally. `EXTERNAL_TEXT_MAX_ROWS` (or `EXTERNAL_MAX_ROWS`) limits a smoke run without changing array-to-manifest alignment; use `0` later to resume the complete manifests. Set `EXTERNAL_TEXT_RETRY_FAILED=1` to retry failed text rows.


In [ ]:
from pathlib import Path
import gc
import hashlib
import json
import os
import re

import numpy as np
import torch
from tqdm.auto import tqdm

ARTIFACT_ROOT = Path(os.environ.get("ECG_CXR_BRIDGE_ARTIFACT_ROOT", "/data/liangz2/ecg_analysis/symile/ecg_cxr_bridge_artifacts"))
config = json.loads((ARTIFACT_ROOT / "config.json").read_text())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TEXT_MAX_ROWS = int(os.environ.get("EXTERNAL_TEXT_MAX_ROWS", os.environ.get("EXTERNAL_MAX_ROWS", "0")))
TEXT_BATCH_SIZE = int(os.environ.get("EXTERNAL_TEXT_BATCH_SIZE", "256"))
RETRY_FAILED = os.environ.get("EXTERNAL_TEXT_RETRY_FAILED", "0").strip().lower() in {"1", "true", "yes"}
if TEXT_MAX_ROWS < 0 or TEXT_BATCH_SIZE < 1:
    raise ValueError("EXTERNAL_TEXT_MAX_ROWS must be nonnegative and EXTERNAL_TEXT_BATCH_SIZE must be positive")
for key in ["text_model_id", "hf_cache"]:
    if key not in config:
        raise KeyError(f"Run notebook 09 first; config is missing {key!r}")

def manifest(modality):
    path = ARTIFACT_ROOT / "external" / modality / "manifest.jsonl"
    if not path.is_file():
        raise FileNotFoundError(f"Run notebook {'10' if modality == 'ecg' else '11'} first; missing {path}")
    values = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    text_key = "machine_diagnosis" if modality == "ecg" else "report_text"
    required = ["sample_id", "split", text_key]
    invalid = [(index, [key for key in required if key not in row or row[key] in (None, "")])
               for index, row in enumerate(values)]
    invalid = [(index, missing) for index, missing in invalid if missing]
    if not values:
        raise ValueError(f"Empty {modality} manifest: {path}")
    if invalid:
        raise KeyError(f"{modality} manifest rows missing required fields: {invalid[:20]}")
    seen, duplicates = set(), set()
    for row in values:
        sample_id = str(row["sample_id"])
        if sample_id in seen:
            duplicates.add(sample_id)
        seen.add(sample_id)
    if duplicates:
        raise ValueError(f"Duplicate {modality} sample IDs: {sorted(duplicates)[:20]}")
    return values

def sequence_sha256(values):
    digest = hashlib.sha256()
    for value in values:
        encoded = str(value).encode()
        digest.update(len(encoded).to_bytes(8, "big")); digest.update(encoded)
    return digest.hexdigest()

rows = {name: manifest(name) for name in ["ecg", "cxr"]}
manifest_hashes = {name: sequence_sha256(
    json.dumps(row, sort_keys=True, separators=(",", ":"), ensure_ascii=False) for row in values)
    for name, values in rows.items()}
print({name: {"rows": len(value), "manifest_sha256": manifest_hashes[name][:12]}
       for name, value in rows.items()}, "device", DEVICE,
      "text_max_rows", TEXT_MAX_ROWS, "retry_failed", RETRY_FAILED)


In [ ]:
ECG_CONCEPT_RULE_VERSION = "ecg_assertion_rules_v3"
CXR_CONCEPT_RULE_VERSION = "cxr_assertion_rules_v2"
ECG_RULES = {
    # Sinus rhythm alone does not imply that the complete ECG is normal.
    "normal_ecg": [
        r"\bnormal(?:\s+12[- ]lead)?\s+(?:ecg|electrocardiogram)\b",
        r"\b(?:ecg|electrocardiogram)\s+(?:is\s+)?within normal limits\b",
    ],
    "atrial_fibrillation": [r"\batrial fibrillation\b", r"\ba[- ]?fib(?:rillation)?\b"],
    "atrial_flutter": [r"\batrial flutter\b"],
    "tachycardia": [r"\btachycard(?:ia|ic)\b", r"\b(?:svt|v[- ]?tach)\b"],
    "bradycardia": [r"\bbradycard(?:ia|ic)\b"],
    "conduction_abnormality": [
        r"\bbundle branch block\b", r"\b(?:rbbb|lbbb|ivcd)\b",
        r"\b(?:av|atrioventricular|heart) block\b",
        r"\b(?:first|second|third)[ -]degree (?:av|atrioventricular) block\b",
        r"\bintraventricular conduction (?:delay|defect)\b",
    ],
    "ventricular_hypertrophy": [
        r"\b(?:left|right) ventricular hypertrophy\b",
        r"\bbiventricular hypertrophy\b", r"\bventricular hypertrophy\b",
        r"\b(?:lvh|rvh)\b",
    ],
    "ischemia": [
        r"\bischemi\w*\b", r"\bst[- ]t abnormalit\w*\b",
        r"\bst(?:[- ]segment)?\s+(?:depression|elevation)\b",
    ],
    "infarction": [
        r"\bmyocardial infarction\b", r"\binfarct(?:ion|ed)?\b",
        r"\b(?:stemi|nstemi)\b",
    ],
    "paced_rhythm": [
        r"\b(?:atrial|ventricular|dual[- ]chamber)?\s*paced rhythm\b",
        r"\bpacing spikes?\b", r"\bpacemaker rhythm\b",
    ],
    "heart_failure": [r"\bheart failure\b", r"\bcongestive heart failure\b"],
    "low_voltage": [r"\blow[- ]voltage\b"],
    "qt_abnormality": [
        r"\b(?:prolonged|long|short)\s+qt(?:c)?(?: interval)?\b",
        r"\bqt(?:c)? prolongation\b",
    ],
}
CXR_RULES = {
    "no_finding": [r"no acute cardiopulmonary", r"no finding", r"normal chest"],
    "cardiomegaly": [r"cardiomegal", r"enlarged cardiomediast"], "edema": [r"\bedema\b", r"vascular congestion"],
    "pleural_effusion": [r"pleural effusion"], "pneumonia": [r"pneumonia"], "consolidation": [r"consolidat"],
    "atelectasis": [r"atelecta"], "pneumothorax": [r"pneumothorax"], "lung_opacity": [r"lung opacity", r"airspace opacity"],
    "support_device": [r"support device", r"pacemaker", r"central line", r"endotracheal tube"],
}

CHEXPERT_LABEL_CONCEPTS = {
    "atelectasis": "atelectasis", "cardiomegaly": "cardiomegaly",
    "consolidation": "consolidation", "edema": "edema",
    "enlarged cardiomediastinum": "enlarged_cardiomediastinum",
    "fracture": "fracture", "lung lesion": "lung_lesion",
    "lung opacity": "lung_opacity", "no finding": "no_finding",
    "pleural effusion": "pleural_effusion", "pleural other": "pleural_other",
    "pneumonia": "pneumonia", "pneumothorax": "pneumothorax",
    "support devices": "support_device", "support device": "support_device",
}
NEGATED_BEFORE = re.compile(
    r"(?:\bno\b|\bwithout\b|\bnegative for\b|\babsence of\b|\bfree of\b|\bdenies\b)"
    r"(?:\W+\w+){0,8}\W*$", re.I)
NEGATED_AFTER = re.compile(r"^\W*(?:is |are |was |were )?(?:not|absent)(?:\W|$)", re.I)
UNCERTAIN_BEFORE = re.compile(
    r"(?:\bpossible\b|\bpossibly\b|\bprobable\b|\blikely\b|\bmay\b|\bmight\b|"
    r"\bcannot exclude\b|\bcannot rule out\b|\bconcerning for\b|\bsuspicious for\b|\bquestion of\b)"
    r"(?:\W+\w+){0,8}\W*$", re.I)
UNCERTAIN_AFTER = re.compile(r"^\W*(?:is |are )?(?:possible|probable|suspected)(?:\W|$)", re.I)

def assertion(text, match, concept):
    if concept == "no_finding":
        return "present"
    before = text[max(0, match.start() - 120):match.start()]
    before = re.split(r"[.;\n]|\b(?:but|however|although|yet)\b", before, flags=re.I)[-1]
    after = text[match.end():match.end() + 50]
    if NEGATED_BEFORE.search(before) or NEGATED_AFTER.search(after):
        return "absent"
    if UNCERTAIN_BEFORE.search(before) or UNCERTAIN_AFTER.search(after):
        return "uncertain"
    return "present"

def rule_concepts(text, rules, labels=()):
    value = " ".join(str(text).split()).lower()
    found = set()
    for concept, patterns in rules.items():
        for pattern in patterns:
            for match in re.finditer(pattern, value):
                found.add(f"{assertion(value, match, concept)}:{concept}")
    if isinstance(labels, dict):
        labels = [key for key, present in labels.items()
                  if present not in (None, False, 0, 0.0, "", "0", "false", "False")]
    elif isinstance(labels, str):
        labels = [labels]
    for label in labels or ():
        concept = CHEXPERT_LABEL_CONCEPTS.get(str(label).strip().lower())
        if concept:
            found.add(f"present:{concept}")
    return sorted(found)

ECG_RULE_TESTS = [
    ("Normal 12-lead ECG.", {"present:normal_ecg"}),
    ("Normal sinus rhythm. Inferior infarct, age undetermined.", {"present:infarction"}),
    ("No evidence of atrial fibrillation.", {"absent:atrial_fibrillation"}),
    ("Possible left ventricular hypertrophy.", {"uncertain:ventricular_hypertrophy"}),
    ("No acute ischemic changes.", {"absent:ischemia"}),
    ("Ventricular paced rhythm.", {"present:paced_rhythm"}),
]
for test_text, expected in ECG_RULE_TESTS:
    observed = set(rule_concepts(test_text, ECG_RULES))
    if observed != expected:
        raise AssertionError(
            f"ECG concept-rule self-test failed for {test_text!r}: "
            f"expected={sorted(expected)}, observed={sorted(observed)}")
print("ECG concept extractor:", ECG_CONCEPT_RULE_VERSION,
      "self-tests:", len(ECG_RULE_TESTS))

def validate_llm_concepts(value, sample_id):
    if not isinstance(value, list):
        raise TypeError(f"LLM concepts for {sample_id} must be a list")
    concepts = []
    for item in value:
        concept = str(item).strip().lower().replace(" ", "_")
        if not re.fullmatch(r"(?:present|absent|uncertain):[a-z0-9][a-z0-9_/-]*", concept):
            raise ValueError(f"Invalid assertion-aware concept for {sample_id}: {item!r}")
        concepts.append(concept)
    return sorted(set(concepts))

llm_path = os.environ.get("LLM_CONCEPTS_JSONL", "")
llm = {}
if llm_path:
    known_ids = {str(row["sample_id"]) for modality_rows in rows.values() for row in modality_rows}
    for line_number, line in enumerate(Path(llm_path).read_text().splitlines(), start=1):
        if not line.strip():
            continue
        item = json.loads(line)
        sample_id = str(item["sample_id"])
        if sample_id not in known_ids:
            raise KeyError(f"Unknown LLM sample_id at line {line_number}: {sample_id}")
        if sample_id in llm:
            raise ValueError(f"Duplicate LLM sample_id: {sample_id}")
        llm[sample_id] = validate_llm_concepts(item["concepts"], sample_id)

text_root = ARTIFACT_ROOT / "external" / "text"
text_root.mkdir(parents=True, exist_ok=True)
prompt_path = text_root / "llm_concept_requests.jsonl"
with prompt_path.open("w") as stream:
    for modality in ["ecg", "cxr"]:
        key = "machine_diagnosis" if modality == "ecg" else "report_text"
        for row in rows[modality]:
            prompt = ("Extract only explicitly stated normalized clinical concepts. Return JSON with sample_id and "
                      "concepts, where each concept is exactly present:<concept>, absent:<concept>, or "
                      "uncertain:<concept>. Preserve negation and uncertainty; do not infer identity, causality, "
                      "or unstated diagnoses. Text: " + row[key])
            stream.write(json.dumps({"sample_id": row["sample_id"], "modality": modality, "prompt": prompt}) + "\n")

text_views, concept_records = {}, {}
for modality in ["ecg", "cxr"]:
    key = "machine_diagnosis" if modality == "ecg" else "report_text"
    concept_rule_version = (ECG_CONCEPT_RULE_VERSION if modality == "ecg"
                            else CXR_CONCEPT_RULE_VERSION)
    raw = [" ".join(str(row[key]).split()) for row in rows[modality]]
    rules = ECG_RULES if modality == "ecg" else CXR_RULES
    concepts = []
    for row in rows[modality]:
        values = llm.get(str(row["sample_id"]))
        if values is None:
            values = rule_concepts(row[key], rules, row.get("labels", []))
        concepts.append(values)
    concept_text = ["; ".join(values) if values else "uncertain:unspecified_clinical_finding" for values in concepts]
    text_views[modality] = {"raw": raw, "concept": concept_text}
    concept_records[modality] = concepts
    out = ARTIFACT_ROOT / "external" / modality
    with (out / "concepts.jsonl").open("w") as stream:
        for row, values, text in zip(rows[modality], concepts, concept_text):
            stream.write(json.dumps({"sample_id": row["sample_id"], "concepts": values, "concept_text": text,
                                     "source": "llm" if str(row["sample_id"]) in llm else "rules",
                                     "concept_rule_version": concept_rule_version}) + "\n")
    print(modality, {"rows": len(raw), "with_concepts": sum(bool(v) for v in concepts),
                     "assertions": {status: sum(any(value.startswith(status + ":") for value in item)
                                               for item in concepts)
                                    for status in ["present", "absent", "uncertain"]}})
print("LLM request file:", prompt_path, "loaded LLM responses:", len(llm))


In [ ]:
from open_clip import create_model_from_pretrained, get_tokenizer

model_result = create_model_from_pretrained(config["text_model_id"], cache_dir=config["hf_cache"])
# open_clip releases differ here: some return (model, preprocess), while
# others return (model, preprocess_train, preprocess_val).
model = model_result[0] if isinstance(model_result, (tuple, list)) else model_result
if model is None or not hasattr(model, "encode_text"):
    raise TypeError(f"Unexpected create_model_from_pretrained result: {type(model_result).__name__}")
tokenizer = get_tokenizer(config["text_model_id"])
model = model.to(DEVICE).eval().requires_grad_(False)
TEXT_ENCODING_VERSION = "biomedclip_text_v3_assertion_aware"

@torch.inference_mode()
def encode_text_batch(texts):
    tokens = tokenizer(texts)
    if hasattr(tokens, "to"):
        tokens = tokens.to(DEVICE)
    encoded = torch.nn.functional.normalize(model.encode_text(tokens), dim=-1).float().cpu().numpy()
    if encoded.ndim != 2 or len(encoded) != len(texts) or not np.isfinite(encoded).all():
        raise ValueError(f"Invalid text encoder output: {encoded.shape}")
    return encoded

def read_json(path, default):
    try:
        return json.loads(path.read_text())
    except (FileNotFoundError, json.JSONDecodeError):
        return default

def read_jsonl(path):
    try:
        return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    except (FileNotFoundError, json.JSONDecodeError):
        return []

def resumable_text_embeddings(modality, view, texts):
    out = ARTIFACT_ROOT / "external" / modality
    feature_path = out / f"text_{view}_biomedclip.npy"
    status_path = out / f"text_{view}_encoding_status.npy"
    metadata_path = out / f"text_{view}_metadata.json"
    error_path = out / f"text_{view}_encoding_errors.jsonl"
    probe = encode_text_batch([texts[0]])
    shape = (len(texts), probe.shape[1])
    identity = {
        "modality": modality, "view": view, "rows": len(texts),
        "manifest_sha256": manifest_hashes[modality],
        "text_sha256": sequence_sha256(texts), "text_model_id": config["text_model_id"],
        "encoding_version": TEXT_ENCODING_VERSION, "output_dim": int(probe.shape[1]),
    }
    old = read_json(metadata_path, {})
    cache_matches = isinstance(old, dict) and all(old.get(key) == value for key, value in identity.items())
    try:
        cache_matches = (cache_matches and feature_path.is_file() and status_path.is_file()
                         and np.load(feature_path, mmap_mode="r").shape == shape
                         and np.load(status_path, mmap_mode="r").shape == (len(texts),))
    except (OSError, ValueError):
        cache_matches = False
    if not cache_matches:
        features = np.lib.format.open_memmap(feature_path, mode="w+", dtype="float32", shape=shape)
        features[:] = np.nan; features.flush()
        status = np.lib.format.open_memmap(status_path, mode="w+", dtype="int8", shape=(len(texts),))
        status[:] = 0; status.flush()
        metadata_path.write_text(json.dumps({**identity, "complete": False}, indent=2))
    features = np.load(feature_path, mmap_mode="r+")
    status = np.load(status_path, mmap_mode="r+")
    eligible = np.arange(min(TEXT_MAX_ROWS, len(texts)) if TEXT_MAX_ROWS > 0 else len(texts))
    pending = eligible[status[eligible] != 1] if RETRY_FAILED else eligible[status[eligible] == 0]
    previous_errors = read_jsonl(error_path)
    errors = []
    for start in tqdm(range(0, len(pending), TEXT_BATCH_SIZE), desc=f"text {modality} {view}"):
        index = pending[start:start + TEXT_BATCH_SIZE]
        batch = [texts[int(value)] for value in index]
        try:
            features[index] = encode_text_batch(batch); status[index] = 1
        except Exception:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            for row_index, text in zip(index, batch):
                try:
                    features[row_index] = encode_text_batch([text])[0]; status[row_index] = 1
                except Exception as exc:
                    status[row_index] = -1
                    errors.append({"row_index": int(row_index), "sample_id": rows[modality][int(row_index)]["sample_id"],
                                   "error": repr(exc)})
        features.flush(); status.flush()
    error_by_index = {
        int(item["row_index"]): item for item in previous_errors
        if 0 <= int(item.get("row_index", -1)) < len(status)
        and status[int(item["row_index"])] == -1
    }
    for item in errors:
        error_by_index[int(item["row_index"])] = item
    current_errors = [error_by_index[index] for index in sorted(error_by_index)
                      if status[index] == -1]
    error_path.write_text("".join(json.dumps(item) + "\n" for item in current_errors))
    summary = {**identity, "selected_rows": len(eligible), "pending_processed": len(pending),
               "valid": int((status == 1).sum()), "failed": int((status == -1).sum()),
               "unprocessed": int((status == 0).sum()), "errors_this_run": len(errors),
               "complete": bool(np.all(status == 1))}
    metadata_path.write_text(json.dumps(summary, indent=2))
    print(summary)

for modality in ["ecg", "cxr"]:
    for view in ["raw", "concept"]:
        resumable_text_embeddings(modality, view, text_views[modality][view])
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
